# Import Library

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

In [ ]:
# Pilihan:
# 0 = Full Dataset
# 1 = 10k
# 2 = 20k
# 3 = 30k
# 4 = 40k

CORPUS_SIZE = 1

if CORPUS_SIZE == 0:
    DATASET_NAME = "full"
elif CORPUS_SIZE == 1:
    DATASET_NAME = "10k"
elif CORPUS_SIZE == 2:
    DATASET_NAME = "20k"
elif CORPUS_SIZE == 3:
    DATASET_NAME = "30k"
elif CORPUS_SIZE == 4:
    DATASET_NAME = "40k"
else:
    raise ValueError(
        "CORPUS_SIZE harus 0, 1, 2, 3, atau 4"
    )

DATA_DIR = Path(
    f"dataset/preprocessed/{DATASET_NAME}"
)

print("="*40)
print("Corpus :", DATASET_NAME)
print("Path   :", DATA_DIR)
print("="*40)

# Load Dataset, Label Encoder & Tokenizer

Dataset

In [ ]:
train_df = pd.read_csv(
    DATA_DIR / "train_bert.csv"
)

test_df = pd.read_csv(
    DATA_DIR / "test_bert.csv"
)

print()

print("Train Shape :", train_df.shape)

print("Test Shape  :", test_df.shape)

display(
    train_df.head()
)

In [ ]:
train_ds = Dataset.from_pandas(
    train_df
)

test_ds = Dataset.from_pandas(
    test_df
)

print()

print(train_ds)

print()

print(test_ds)

Label Encoder

In [ ]:
le = joblib.load(
    "dataset/preprocessed/label_encoder.pkl"
)

class_names = list(
    le.classes_
)

NUM_LABELS = len(
    class_names
)

print()

print("Class Names:")

print(class_names)

print()

print("Number of Classes:")

print(NUM_LABELS)

Tokenizer

In [ ]:

MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print()

print("Tokenizer Loaded Successfully!")

# Tokenization

In [ ]:
def tokenize(batch):

    return tokenizer(

        batch["text"],

        truncation=True,

        padding="max_length",

        max_length=128

    )

train_ds = train_ds.map(

    tokenize,

    batched=True

)

test_ds = test_ds.map(

    tokenize,

    batched=True

)

print()

print("Tokenization Completed!")

In [ ]:
train_ds.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "label"

    ]

)

test_ds.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "label"

    ]

)

print()

print("Dataset Format Ready!")

# DL Things

## Load Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=NUM_LABELS

)

print()

print("Model Loaded!")

print()

print(model.config)

## Compute Metrics

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(

        logits,

        axis=1

    )

    return {

        "accuracy":

        accuracy_score(

            labels,

            preds

        ),

        "precision":

        precision_score(

            labels,

            preds,

            average="macro",

            zero_division=0

        ),

        "recall":

        recall_score(

            labels,

            preds,

            average="macro",

            zero_division=0

        ),

        "f1":

        f1_score(

            labels,

            preds,

            average="macro",

            zero_division=0

        )

    }

print()

print("Metric Function Ready!")

## Training Arguments

In [ ]:
training_args = TrainingArguments(

    output_dir=f"models/skenario 4/checkpoints/{DATASET_NAME}",

    eval_strategy="epoch",

    save_strategy="epoch",

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100,

    logging_dir=f"models/skenario 4/logs/{DATASET_NAME}",

    report_to="none"

)

print()

print("Training Arguments Ready!")

## Trainer

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=test_ds,

    compute_metrics=compute_metrics

)

print()

print("Trainer Ready!")

# Modeling

Training

In [ ]:
train_result = trainer.train()

print()

print("Training Completed!")

Predict

In [ ]:
predictions = trainer.predict(
    test_ds
)

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

y_true = predictions.label_ids

print()

print("Prediction Completed!")

print()

print("Total Test Data :", len(y_true))

# Evaluation

## Confussion Matrix

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(
    figsize=(6,5)
)

sns.heatmap(

    cm,

    annot=True,

    fmt="d",

    cmap="Blues",

    xticklabels=class_names,

    yticklabels=class_names

)

plt.xlabel(
    "Predicted Label"
)

plt.ylabel(
    "True Label"
)

plt.title(
    f"Confusion Matrix - IndoBERT ({DATASET_NAME})"
)

plt.tight_layout()

plt.show()

## Classificattion Report

In [ ]:
print()

print("=== Classification Report ===")

print()

print(

    classification_report(

        y_true,

        y_pred,

        target_names=class_names,

        zero_division=0

    )

)

## Overall

In [ ]:
acc = accuracy_score(
    y_true,
    y_pred
)

prec = precision_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

rec = recall_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

print()

print("=== IndoBERT ===")

print()

print(f"Accuracy : {acc:.4f}")

print(f"Precision: {prec:.4f}")

print(f"Recall   : {rec:.4f}")

print(f"F1-Score : {f1:.4f}")

In [ ]:
result = pd.DataFrame([

    {

        "Train_Size": len(train_df),

        "Accuracy": acc,

        "Precision": prec,

        "Recall": rec,

        "F1-Score": f1

    }

])

result

# Save Train Result

In [ ]:
os.makedirs(
    "results/training",
    exist_ok=True
)

RESULT_FILE = "results/training/indobert_results.csv"

if os.path.exists(
    RESULT_FILE
):

    previous = pd.read_csv(
        RESULT_FILE
    )

    result = pd.concat(

        [

            previous,

            result

        ],

        ignore_index=True

    )

    result = result.drop_duplicates(

        subset=["Train_Size"],

        keep="last"

    )

result = result.sort_values(
    by="Train_Size"
)

result.to_csv(

    RESULT_FILE,

    index=False

)

print()

print(result)

# Save Model

In [ ]:
SAVE_PATH = f"models/skenario 4/indobert_{DATASET_NAME}"

os.makedirs(
    SAVE_PATH,
    exist_ok=True
)

trainer.save_model(
    SAVE_PATH
)

print()

print("Best Model Saved!")

print()

print(SAVE_PATH)

# Save Tokenizer

In [ ]:
tokenizer.save_pretrained(
    SAVE_PATH
)

print()

print("Tokenizer Saved!")

print()

print(SAVE_PATH)